# MFAR — Run All Stages 01–07\n\nJalankan satu sel di bawah. Pipeline menimpa artefak bernama sama dan menghasilkan CSV/JSON, peta HTML, dashboard interaktif, serta workbook Excel ringkas.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import subprocess
import sys
from pathlib import Path

BRANCH = "test-colab-pipeline"
REPO_URL = "https://github.com/yanto-mashardi/MFAR_Modular_Colab_Pipeline.git"
REPO = Path("/content/MFAR_Modular_Colab_Pipeline")
DRIVE_ROOT = Path("/content/drive/MyDrive/Post Doctor/In_Out_MFAR_Modular_Colab_Pipeline")

if not REPO.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "switch", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True)

os.environ["MFAR_GDRIVE_ROOT"] = str(DRIVE_ROOT)
os.environ["MFAR_CODE_ROOT"] = str(REPO)
os.environ["PYTHONPATH"] = f"{REPO}:{os.environ.get('PYTHONPATH', '')}"

required = [
    DRIVE_ROOT / "data_raw" / "ais_raw.csv",
    DRIVE_ROOT / "data_raw" / "vehicle_arrival_rate_30min.csv",
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Input Drive tidak ditemukan:\n- " + "\n- ".join(missing))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")],
    check=True,
)

notebooks = [
    "01_AIS_Input_and_Cleaning.ipynb",
    "02_Time_Grid_and_State_Preparation.ipynb",
    "03_Monitoring_State.ipynb",
    "04_No_Intervention_Forecast.ipynb",
    "05_Fuzzification.ipynb",
    "06_Rule_Evaluation.ipynb",
    "07_Candidate_Action.ipynb",
]
executed_dir = DRIVE_ROOT / "executed_notebooks"
executed_dir.mkdir(parents=True, exist_ok=True)

for stage, notebook in enumerate(notebooks, 1):
    print(f"\n{'='*72}\nSTAGE {stage:02d}: {notebook}\n{'='*72}", flush=True)
    subprocess.run(
        [
            "jupyter", "nbconvert", "--to", "notebook", "--execute",
            str(REPO / "notebooks" / notebook),
            "--output", notebook,
            "--output-dir", str(executed_dir),
            "--ExecutePreprocessor.timeout=-1",
            "--ExecutePreprocessor.kernel_name=python3",
        ],
        cwd=REPO,
        env=os.environ.copy(),
        check=True,
    )
    print(f"STAGE {stage:02d} BERHASIL", flush=True)

print("\nSELURUH STAGE 01–07 BERHASIL")
print("Data teknis :", DRIVE_ROOT / "stage_output")
print("Notebook run:", executed_dir)
print("Buka file HTML/XLSX pada folder stage masing-masing untuk membaca hasil.")
